# DeepSD Workflow: Stacked SRCNN for Climate Downscaling

Educational notebook accompanying the review of **Vandal et al. (2017)**, *DeepSD: Generating High Resolution Climate Change Projections through Single Image Super-Resolution* (ACM SIGKDD).

## Learning goals
1. Represent precipitation + elevation as multi-channel "images".
2. Implement an augmented SRCNN stage (upsample + topography + 9x9/1x1/5x5 CNN).
3. Stack three 2x stages for an overall **8x** resolution gain.
4. Compare against a bilinear / BCSD-like baseline using bias, correlation, RMSE, and extremes.
5. Export artifacts for the R diagnostics script.

> This demo uses **synthetic** fields so it runs without PRISM/GTOPO30 downloads. Replace arrays with real gridded data for research use.


## 1. Setup and reproducibility


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 2017
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUTPUT_DIR = Path("outputs")
MODEL_DIR = Path("models")
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

print("device:", device)
print("torch:", torch.__version__)


## 2. Synthetic precipitation + elevation (PRISM / GTOPO30 analogue)

High-resolution precipitation depends on synoptic moisture, **orographic enhancement**, mesoscale structure, and wet/dry sparsity—mirroring challenges highlighted in the paper.


In [ ]:
def block_average(arr, factor):
    h, w = arr.shape
    return arr.reshape(h // factor, factor, w // factor, factor).mean(axis=(1, 3))


class SyntheticPrecipDataset(Dataset):
    def __init__(self, n_days, high_h=64, high_w=128, stages=3, seed=0):
        self.stages = stages
        rng = np.random.default_rng(seed)
        yy, xx = np.meshgrid(
            np.linspace(0, 1, high_h), np.linspace(0, 1, high_w), indexing="ij"
        )
        elevation = (
            0.55 * np.exp(-((xx - 0.25) ** 2 + (yy - 0.45) ** 2) / 0.03)
            + 0.35 * np.exp(-((xx - 0.70) ** 2 + (yy - 0.60) ** 2) / 0.05)
            + 0.15 * yy
        )
        elevation = (elevation - elevation.min()) / (elevation.max() - elevation.min() + 1e-8)
        self.elevation = elevation.astype(np.float32)

        factor = 2 ** stages
        xs, ys, es = [], [], []
        for _ in range(n_days):
            phase = rng.uniform(0, 2 * np.pi)
            synoptic = np.maximum(
                0.0,
                6
                + 4 * np.sin(2 * np.pi * xx + phase)
                + 3 * np.cos(2 * np.pi * yy - 0.5 * phase),
            )
            oro = 5.0 * elevation * (1 + 0.3 * np.sin(4 * np.pi * xx))
            meso = 1.2 * np.abs(np.sin(10 * np.pi * xx) * np.cos(8 * np.pi * yy))
            noise = rng.normal(0, 0.35, size=(high_h, high_w))
            wet = rng.random((high_h, high_w)) > 0.35
            high = np.clip((synoptic + oro + meso + noise) * wet, 0, None).astype(np.float32)
            low = block_average(high, factor).astype(np.float32)
            xs.append(low[None, ...])
            ys.append(high[None, ...])
            es.append(elevation[None, ...])

        self.x = np.stack(xs)
        self.y = np.stack(ys)
        self.e = np.stack(es)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.x[idx]),
            torch.from_numpy(self.e[idx]),
            torch.from_numpy(self.y[idx]),
        )


STAGES = 3  # 2x * 2x * 2x = 8x overall (paper: 1.0 deg -> 1/8 deg)
HIGH_H, HIGH_W = 64, 128
LOW_H, LOW_W = HIGH_H // (2 ** STAGES), HIGH_W // (2 ** STAGES)

train_ds = SyntheticPrecipDataset(180, HIGH_H, HIGH_W, STAGES, seed=SEED)
val_ds = SyntheticPrecipDataset(40, HIGH_H, HIGH_W, STAGES, seed=SEED + 1)
test_ds = SyntheticPrecipDataset(40, HIGH_H, HIGH_W, STAGES, seed=SEED + 2)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False)

print(f"LR grid: {LOW_H}x{LOW_W}  ->  HR grid: {HIGH_H}x{HIGH_W}  ({2**STAGES}x)")
print(len(train_ds), len(val_ds), len(test_ds))


### Visualize one sample day


In [ ]:
x0, e0, y0 = test_ds[0]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].imshow(x0[0].numpy(), cmap="Blues")
axes[0].set_title(f"LR precip ({LOW_H}x{LOW_W})")
axes[1].imshow(e0[0].numpy(), cmap="terrain")
axes[1].set_title("Elevation (aux channel)")
axes[2].imshow(y0[0].numpy(), cmap="Blues")
axes[2].set_title(f"HR precip ({HIGH_H}x{HIGH_W})")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. Augmented SRCNN and stacked DeepSD

Paper architecture per stage (Dong et al. / DeepSD):
- Layer 1: 64 filters, 9x9, ReLU
- Layer 2: 32 filters, 1x1, ReLU
- Layer 3: 1 filter, 5x5 (reconstruction)

DeepSD difference vs vanilla SR: concatenate **upsampled LR precipitation** with **HR elevation** before the CNN.


In [ ]:
def bilinear_upsample(x, scale):
    return nn.functional.interpolate(
        x, scale_factor=scale, mode="bilinear", align_corners=False
    )


class SRCNN(nn.Module):
    def __init__(self, scale=2, in_channels=2):
        super().__init__()
        self.scale = scale
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=9, padding=4),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, kernel_size=5, padding=2),
        )

    def forward(self, precip_lr, elev_hr):
        precip_up = bilinear_upsample(precip_lr, self.scale)
        if elev_hr.shape[-2:] != precip_up.shape[-2:]:
            elev_hr = nn.functional.interpolate(
                elev_hr, size=precip_up.shape[-2:], mode="bilinear", align_corners=False
            )
        return self.net(torch.cat([precip_up, elev_hr], dim=1))


class DeepSD(nn.Module):
    def __init__(self, stages=3):
        super().__init__()
        self.stages = nn.ModuleList([SRCNN(scale=2) for _ in range(stages)])

    def forward(self, precip_lr, elev_hr):
        x = precip_lr
        for i, stage in enumerate(self.stages):
            th = precip_lr.shape[-2] * (2 ** (i + 1))
            tw = precip_lr.shape[-1] * (2 ** (i + 1))
            elev_i = nn.functional.interpolate(
                elev_hr, size=(th, tw), mode="bilinear", align_corners=False
            )
            x = torch.relu(stage(x, elev_i))
        return x


model = DeepSD(stages=STAGES).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print("trainable parameters:", n_params)


## 4. Train with MSE loss (paper Eq. 1)


In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 12
history = []


def run_epoch(loader, train=True):
    model.train(train)
    losses = []
    for precip_lr, elev, precip_hr in loader:
        precip_lr = precip_lr.to(device)
        elev = elev.to(device)
        precip_hr = precip_hr.to(device)
        if train:
            optimizer.zero_grad()
        pred = model(precip_lr, elev)
        loss = criterion(pred, precip_hr)
        if train:
            loss.backward()
            optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses))


best_val = float("inf")
best_path = MODEL_DIR / "deepsd_notebook_best.pt"
for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)
    history.append({"epoch": epoch, "train_mse": tr, "val_mse": va})
    print(f"Epoch {epoch:02d} | train={tr:.4f} | val={va:.4f}")
    if va < best_val:
        best_val = va
        torch.save(model.state_dict(), best_path)

try:
    state = torch.load(best_path, map_location=device, weights_only=True)
except TypeError:
    state = torch.load(best_path, map_location=device)
model.load_state_dict(state)
pd.DataFrame(history).to_csv(OUTPUT_DIR / "notebook_training_history.csv", index=False)
pd.DataFrame(history).plot(
    x="epoch", y=["train_mse", "val_mse"], marker="o", figsize=(7, 4)
)
plt.ylabel("MSE")
plt.title("DeepSD training curves")
plt.grid(True, alpha=0.3)
plt.show()


## 5. Evaluate vs bilinear baseline (BCSD-like spatial step)

Paper metrics: **bias**, **Pearson correlation**, **RMSE**, and distributional **skill**. We also probe extreme percentiles as in Figure 5.


In [ ]:
@torch.no_grad()
def collect_predictions(loader):
    model.eval()
    yt, yp, yb = [], [], []
    for precip_lr, elev, precip_hr in loader:
        precip_lr = precip_lr.to(device)
        elev = elev.to(device)
        pred = model(precip_lr, elev)
        scale = pred.shape[-1] // precip_lr.shape[-1]
        baseline = bilinear_upsample(precip_lr, scale)
        yt.append(precip_hr.numpy())
        yp.append(pred.cpu().numpy())
        yb.append(baseline.cpu().numpy())
    return np.concatenate(yt), np.concatenate(yp), np.concatenate(yb)


def metrics(y_true, y_pred):
    yt, yp = y_true.ravel(), y_pred.ravel()
    bias = float(np.mean(yp - yt))
    corr = float(np.corrcoef(yt, yp)[0, 1])
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    mae = float(mean_absolute_error(yt, yp))
    bins = np.linspace(0, max(yt.max(), yp.max()) + 1e-6, 51)
    ho, _ = np.histogram(yt, bins=bins, density=True)
    hm, _ = np.histogram(yp, bins=bins, density=True)
    skill = float(np.minimum(ho, hm).sum() / max(ho.sum(), 1e-8))
    return dict(
        bias=bias,
        corr=corr,
        rmse=rmse,
        mae=mae,
        skill=skill,
        mean=float(np.mean(yt)),
        median=float(np.median(yt)),
        sd=float(np.std(yt)),
    )


y_true, y_pred, y_base = collect_predictions(test_loader)
summary = pd.DataFrame(
    [
        {"model": "DeepSD", **metrics(y_true, y_pred)},
        {"model": "BilinearBaseline", **metrics(y_true, y_base)},
    ]
)
display(summary)
summary.to_csv(OUTPUT_DIR / "notebook_metrics_summary.csv", index=False)


In [ ]:
# Extreme precipitation thresholds (paper Fig. 5 style)
rows = []
for p in [90, 95, 99]:
    thr = np.percentile(y_true, p)
    mask = y_true.ravel() >= thr
    rows.append(
        {
            "percentile": p,
            "model": "DeepSD",
            **metrics(y_true.ravel()[mask], y_pred.ravel()[mask]),
        }
    )
    rows.append(
        {
            "percentile": p,
            "model": "BilinearBaseline",
            **metrics(y_true.ravel()[mask], y_base.ravel()[mask]),
        }
    )
extreme_df = pd.DataFrame(rows)
display(extreme_df)
extreme_df.to_csv(OUTPUT_DIR / "notebook_extreme_metrics.csv", index=False)

pivot = extreme_df.pivot(index="percentile", columns="model", values="rmse")
pivot.plot(kind="bar", figsize=(7, 4), rot=0)
plt.ylabel("RMSE")
plt.title("Extreme-event RMSE by percentile")
plt.grid(axis="y", alpha=0.3)
plt.show()


## 6. Spatial diagnostics and export for R


In [ ]:
idx = 0
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
panels = [
    (test_ds.e[idx, 0], "Elevation", "terrain"),
    (y_true[idx, 0], "True HR", "Blues"),
    (y_pred[idx, 0], "DeepSD", "Blues"),
    (y_base[idx, 0], "Bilinear", "Blues"),
    (y_pred[idx, 0] - y_true[idx, 0], "DeepSD error", "RdBu_r"),
    (y_base[idx, 0] - y_true[idx, 0], "Baseline error", "RdBu_r"),
]
for ax, (arr, title, cmap) in zip(axes.ravel(), panels):
    im = ax.imshow(arr, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("DeepSD notebook demo — one test day")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "notebook_overview.png", dpi=140)
plt.show()

pred_df = pd.DataFrame(
    {
        "y_true": y_true.ravel(),
        "y_pred": y_pred.ravel(),
        "y_baseline": y_base.ravel(),
    }
)
if len(pred_df) > 20000:
    pred_df = pred_df.sample(20000, random_state=SEED)
pred_df.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
print("Wrote", OUTPUT_DIR / "predictions.csv")
print("Next: Rscript deepsd_analysis.R outputs/predictions.csv outputs")


## 7. Interpretation notes (tie-back to the paper)

| Paper claim | What this notebook illustrates |
|---|---|
| Stacked 2x SRCNNs for 8x downscaling | `DeepSD` with `STAGES=3` |
| Elevation as auxiliary HR channel | `torch.cat([precip_up, elev])` |
| Outperforms simple spatial baselines on RMSE/corr | metrics table + extremes |
| Precipitation sparsity / extremes matter | wet-mask synthesis + percentile RMSE |
| Scalable feed-forward inference | single forward pass after training |

**Limitations of this demo (also noted by authors for real DeepSD):** synthetic stationarity, no ESM bias-correction step, no Bayesian uncertainty, and no transfer test to unseen regions.

### Suggested extensions
- Replace synthetic arrays with PRISM + GTOPO30 NetCDF tiles.
- Train each SRCNN stage independently (as in the original paper) then freeze/stack.
- Add quantile mapping before spatial disaggregation for a fuller BCSD baseline.
- Explore dropout / deep ensembles for uncertainty (Gal, 2016 — cited in paper).
